# NB5 — full evaluation of four trained adapters
Select T4 GPU and Run all. In Setup, upload NB5_INPUT.zip generated beside this notebook.
Uses frozen baseline target=0.84 and full eval. No training or baseline reruns.
The verdict covers correct on target/regression/format/latency; contrasts are ranked by target.
Final cell downloads nb5_results.zip. If interrupted after verdict, download partial results before resetting runtime.


In [ ]:
import hashlib, os, pathlib, subprocess, sys, zipfile
import torch
from google.colab import files
assert torch.cuda.is_available(), "Select T4 GPU before running"
print("GPU:", torch.cuda.get_device_name(0))
print("Choose NB5_INPUT.zip from your local colab folder")
uploaded = files.upload()
assert "NB5_INPUT.zip" in uploaded, "Upload the generated NB5_INPUT.zip"
payload = uploaded["NB5_INPUT.zip"]
assert hashlib.sha256(payload).hexdigest() == '0955858bad758ce69f833b4785d0581eebf538fa095ddd7d5a79c42a8e9376fa', "Wrong input package"
root = pathlib.Path("/content/lab21_nb5_portable")
assert not (root / "results/verdict.json").exists(), "Already evaluated: download results before another run"
root.mkdir(exist_ok=True)
with zipfile.ZipFile("NB5_INPUT.zip") as archive:
    archive.extractall(root)
del uploaded, payload
os.chdir(root)
os.environ["COMPUTE_TIER"] = "T4"
os.environ.pop("BASE_MODEL", None)
os.environ.pop("EVAL_LIMIT", None)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


In [ ]:
import csv, json, subprocess, sys, pathlib
root = pathlib.Path.cwd()
with (root / "results/runs.csv").open() as fh:
    rows = {r["run"]: r for r in csv.DictReader(fh)}
for key in ("correct", "attn_only", "wrong_lr", "qlora"):
    assert int(rows[key]["max_steps"]) == int(rows[key]["actual_steps"]) == 30
subprocess.run([sys.executable, "-u", "scripts/colab_run.py", "nb5"], check=True)


In [ ]:
import json, pathlib, shutil
from google.colab import files
root = pathlib.Path.cwd()
verdict = json.loads((root / "results/verdict.json").read_text())
ranking = json.loads((root / "results/target_ranking.json").read_text())
qualitative = json.loads((root / "results/qualitative_selected.json").read_text())
assert len(ranking) == 4, "Not all adapters evaluated"
assert len(qualitative["examples"]) >= 5
print(json.dumps(verdict, ensure_ascii=False, indent=2))
print(json.dumps(ranking, ensure_ascii=False, indent=2))
print("Real FT losses vs baseline (b):", qualitative["n_losses"])
shutil.make_archive("nb5_results", "zip", "results")
files.download("nb5_results.zip")
print("Evaluation complete. PASS and FAIL are both valid measured outcomes.")
